# Imports

In [1]:
import optuna
import pickle

import numpy as np
import pandas as pd

from tqdm import tqdm

from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict

/home/junior/Documentos/GitHub/kaggle-competition-predicting-student-health/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Utils

In [2]:
def load_pickle(file_path):
    with open(file_path, 'rb') as file:
        return pickle.load(file)

# Loading Datasets

In [3]:
X_train = pd.read_parquet('../data/processed/X_train_stacking_layer_one.parquet')
y_train = pd.read_parquet('../data/interim/y_train.parquet')

X_test = pd.read_parquet('../data/processed/X_test_stacking_layer_one.parquet')

In [4]:
X_train.head()

,lgbm_0,lgbm_1,lgbm_2,xgb_0,xgb_1,xgb_2,cat_0,cat_1,cat_2
0,0.004831,0.002142,0.993027,0.005220,0.002128,0.992652,0.005655,0.000463,0.993882
1,0.998229,0.000143,0.001628,0.997774,0.000322,0.001904,0.994638,0.000368,0.004994
2,0.003982,0.000584,0.995434,0.002362,0.000710,0.996928,0.002327,0.000919,0.996754
3,0.007424,0.001954,0.990622,0.002922,0.003628,0.993450,0.003433,0.002374,0.994194
4,0.993583,0.000012,0.006405,0.997151,0.000013,0.002836,0.994112,0.000005,0.005883


In [5]:
X_test.head()

,lgbm_0,lgbm_1,lgbm_2,xgb_0,xgb_1,xgb_2,cat_0,cat_1,cat_2
0,0.011986,0.002516,0.985498,0.009749,0.003956,0.986295,0.008185,0.005526,0.986288
1,0.469590,0.002922,0.527488,0.344531,0.002104,0.653365,0.530403,0.002094,0.467503
2,0.998492,0.000991,0.000517,0.998555,0.001132,0.000313,0.996306,0.003216,0.000478
3,0.986619,0.000009,0.013372,0.978274,0.000024,0.021702,0.990683,0.000093,0.009224
4,0.008520,0.001975,0.989505,0.005601,0.002080,0.992319,0.003507,0.001947,0.994546


# Machine Learning

In [6]:
label_encoder = load_pickle('../artifacts/label_encoder.pkl')

In [7]:
cols_use = ['cat_0', 'cat_1', 'cat_2']

In [8]:
def objective(trial, X, y):

    w0 = trial.suggest_float('weight_class_0', 0.0, 10.0, log=False)
    w1 = trial.suggest_float('weight_class_1', 0.0, 10.0, log=False)
    w2 = trial.suggest_float('weight_class_2', 0.0, 10.0, log=False)
    weights = np.array([w0, w1, w2])

    proba = X.loc[:, cols_use].to_numpy()
    
    weighted_probas = proba * weights
    pred = np.argmax(weighted_probas, axis=1)
    
    score = balanced_accuracy_score(y, pred)

    return score


study = optuna.create_study(
    direction="maximize", 
    sampler=optuna.samplers.TPESampler(seed=42), 
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=2)
)

study.optimize(
    lambda trial: objective(trial, X_train, y_train.health_condition), 
    n_trials=2000, 
    n_jobs=-1, 
    show_progress_bar=True
)

print("\nBest trial score:")
print(study.best_trial.value)

print("\nBest params:")
print(study.best_trial.params)

[I 2026-07-08 13:32:35,717] A new study created in memory with name: no-name-bc2e9180-4698-4d31-8cb7-031ce4286e80
                                                                                                                                                                                                                   

[I 2026-07-08 13:32:35,876] Trial 7 finished with value: 0.9128511430403047 and parameters: {'weight_class_0': 4.300377413443663, 'weight_class_1': 8.766256680198415, 'weight_class_2': 6.772197382686124}. Best is trial 7 with value: 0.9128511430403047.
[I 2026-07-08 13:32:35,883] Trial 11 finished with value: 0.949077759307233 and parameters: {'weight_class_0': 0.5903968610470989, 'weight_class_1': 9.572000138935445, 'weight_class_2': 8.88297755793739}. Best is trial 11 with value: 0.949077759307233.
[I 2026-07-08 13:32:35,887] Trial 8 finished with value: 0.8530178058835266 and parameters: {'weight_class_0': 7.666224939193928, 'weight_class_1': 4.4402309306423415, 'weight_class_2': 3.9413886943357523}. Best is trial 11 with value: 0.949077759307233.
[I 2026-07-08 13:32:35,893] Trial 5 finished with value: 0.8670943688371734 and parameters: {'weight_class_0': 7.46563800755879, 'weight_class_1': 3.6848184965121913, 'weight_class_2': 8.370166635898837}. Best is trial 11 with value: 0.949

Best trial: 11. Best value: 0.949078:   1%|█▋                                                                                                                                    | 25/2000 [00:00<00:42, 46.78it/s]

[I 2026-07-08 13:32:36,041] Trial 13 finished with value: 0.8980058994336458 and parameters: {'weight_class_0': 0.28180424509909513, 'weight_class_1': 0.15013756803684242, 'weight_class_2': 0.9444942828120428}. Best is trial 11 with value: 0.949077759307233.
[I 2026-07-08 13:32:36,062] Trial 14 finished with value: 0.9468625322622404 and parameters: {'weight_class_0': 0.013955393758803503, 'weight_class_1': 0.407796354541496, 'weight_class_2': 0.6474532016269485}. Best is trial 11 with value: 0.949077759307233.
[I 2026-07-08 13:32:36,065] Trial 15 finished with value: 0.8642786951452398 and parameters: {'weight_class_0': 2.3585565616202473, 'weight_class_1': 0.08419346849631903, 'weight_class_2': 6.390241314366317}. Best is trial 11 with value: 0.949077759307233.
[I 2026-07-08 13:32:36,089] Trial 16 finished with value: 0.9354915631332666 and parameters: {'weight_class_0': 2.200849976841212, 'weight_class_1': 6.564899077760393, 'weight_class_2': 6.2692018389356665}. Best is trial 11 wi

[I 2026-07-08 13:32:36,268] Trial 26 finished with value: 0.9242677313581954 and parameters: {'weight_class_0': 3.608016405949927, 'weight_class_1': 7.696715044355569, 'weight_class_2': 7.877402503139606}. Best is trial 11 with value: 0.949077759307233.
[I 2026-07-08 13:32:36,280] Trial 27 finished with value: 0.9266698395224485 and parameters: {'weight_class_0': 3.776085551326833, 'weight_class_1': 9.919652247268935, 'weight_class_2': 7.775091531249599}. Best is trial 11 with value: 0.949077759307233.
[I 2026-07-08 13:32:36,302] Trial 28 finished with value: 0.9252898529988594 and parameters: {'weight_class_0': 3.593038947276659, 'weight_class_1': 7.888389707942375, 'weight_class_2': 7.954100785421646}. Best is trial 11 with value: 0.949077759307233.
[I 2026-07-08 13:32:36,315] Trial 29 finished with value: 0.928261361415213 and parameters: {'weight_class_0': 3.7085289017791725, 'weight_class_1': 9.863447602176318, 'weight_class_2': 7.998990130682921}. Best is trial 11 with value: 0.9

...

[I 2026-07-08 13:33:20,631] Trial 1986 finished with value: 0.9482582801377392 and parameters: {'weight_class_0': 1.1651137875901427, 'weight_class_1': 9.39966100227459, 'weight_class_2': 8.015746653408016}. Best is trial 1957 with value: 0.9492629103474223.
[I 2026-07-08 13:33:20,662] Trial 1987 finished with value: 0.946925608010471 and parameters: {'weight_class_0': 1.468909675866017, 'weight_class_1': 9.162103908419683, 'weight_class_2': 8.091488493537256}. Best is trial 1957 with value: 0.9492629103474223.
[I 2026-07-08 13:33:20,694] Trial 1988 finished with value: 0.947245018650761 and parameters: {'weight_class_0': 1.3965934639552726, 'weight_class_1': 9.19267120105116, 'weight_class_2': 8.045790080735394}. Best is trial 1957 with value: 0.9492629103474223.
[I 2026-07-08 13:33:20,708] Trial 1989 finished with value: 0.9469930117431459 and parameters: {'weight_class_0': 1.4436441276922123, 'weight_class_1': 9.14691235258784, 'weight_class_2': 7.977686270167525}. Best is trial 195

Best trial: 1957. Best value: 0.949263: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▊| 1998/2000 [00:45<00:00, 43.77it/s]

[I 2026-07-08 13:33:20,852] Trial 1995 finished with value: 0.8960131178524339 and parameters: {'weight_class_0': 6.292620847587896, 'weight_class_1': 9.15564839201461, 'weight_class_2': 7.926684213198196}. Best is trial 1957 with value: 0.9492629103474223.
[I 2026-07-08 13:33:20,870] Trial 1997 finished with value: 0.949167161835159 and parameters: {'weight_class_0': 0.741192517094062, 'weight_class_1': 9.136596386519143, 'weight_class_2': 7.82799989333556}. Best is trial 1957 with value: 0.9492629103474223.
[I 2026-07-08 13:33:20,876] Trial 1996 finished with value: 0.9491605260881834 and parameters: {'weight_class_0': 0.7454272586308692, 'weight_class_1': 9.192649992882156, 'weight_class_2': 7.842471747441204}. Best is trial 1957 with value: 0.9492629103474223.
[I 2026-07-08 13:33:20,881] Trial 1998 finished with value: 0.9482631086882273 and parameters: {'weight_class_0': 1.159097291083694, 'weight_class_1': 9.81337953020939, 'weight_class_2': 7.837827394092884}. Best is trial 1957

Best trial: 1957. Best value: 0.949263: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2000/2000 [00:45<00:00, 44.27it/s]

[I 2026-07-08 13:33:20,891] Trial 1999 finished with value: 0.9491563601727012 and parameters: {'weight_class_0': 0.7472281813207152, 'weight_class_1': 9.806990513808321, 'weight_class_2': 7.792860884760827}. Best is trial 1957 with value: 0.9492629103474223.

Best trial score:
0.9492629103474223

Best params:
{'weight_class_0': 0.6878593354026348, 'weight_class_1': 9.617609913172789, 'weight_class_2': 7.846235980568103}


In [9]:
weights = np.array([study.best_params['weight_class_0'], study.best_params['weight_class_1'], study.best_params['weight_class_2']])
weighted_probas = X_test.loc[:, cols_use].to_numpy() * weights

pred = np.argmax(weighted_probas, axis=1)

In [10]:
sub_labels = label_encoder.inverse_transform(pred)

# Submission

In [11]:
submission = pd.read_csv('../data/submission/sample_submission.csv')
submission['health_condition'] = sub_labels

submission.to_csv('../data/submission/submission_from_5.2.0_catboost_submission.csv', index=False)

In [12]:
submission.head()

,id,health_condition
0,690088,unhealthy
1,690089,unhealthy
2,690090,at-risk
3,690091,at-risk
4,690092,unhealthy
